# NLP Task 2: Temporal Integration and Combined Model

This notebook integrates daily NLP features with financial features, applies a chronological train/validation/test split, and evaluates combined models against the financial-only baselines.

## 1. Imports

In [1]:
import json
from datetime import timedelta
from pathlib import Path

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, roc_auc_score
from xgboost import XGBClassifier

from aggregate import aggregate_daily

## 2. Load Data

In [2]:
BASE_DIR = Path("/content")

ARTICLE_PATH = BASE_DIR / "article_features.csv"
JISDOR_PATH = BASE_DIR / "jisdor_cleaned.xlsx"
FINANCIAL_PATH = BASE_DIR / "financial_features.csv"

articles = pd.read_csv(ARTICLE_PATH)

jisdor = pd.read_excel(
    JISDOR_PATH,
    sheet_name="Informasi Kurs Jisdor",
    header=4,
    usecols=[0, 1, 2]
)

financial = pd.read_csv(FINANCIAL_PATH)

print("Article rows:", len(articles))
print("JISDOR rows:", len(jisdor))
print("Financial rows:", len(financial))

Article rows: 3577
JISDOR rows: 1202
Financial rows: 1181


## 3. Temporal Alignment

In [3]:
articles["date"] = pd.to_datetime(
    articles["date"],
    errors="coerce"
).dt.normalize()

jisdor["date_clean"] = pd.to_datetime(
    jisdor["date_clean"],
    errors="coerce"
).dt.normalize()

jisdor["usd_idr"] = pd.to_numeric(
    jisdor["usd_idr"],
    errors="coerce"
)

jisdor = (
    jisdor
    .dropna(subset=["date_clean"])
    .sort_values("date_clean")
    .drop_duplicates("date_clean", keep="last")
    .reset_index(drop=True)
)

jisdor_dates = set(jisdor["date_clean"].dt.date)
last_jisdor_date = max(jisdor_dates)

print("Missing article dates:", articles["date"].isna().sum())
print("First article date:", articles["date"].min())
print("Last article date:", articles["date"].max())
print("Last JISDOR date:", jisdor["date_clean"].max())

Missing article dates: 0
First article date: 2021-09-01 00:00:00
Last article date: 2026-09-01 00:00:00
Last JISDOR date: 2026-09-01 00:00:00


In [4]:
def get_trade_date(article_date):
    if pd.isna(article_date):
        return pd.NaT

    current_date = article_date.date()

    while current_date <= last_jisdor_date:
        if current_date in jisdor_dates:
            return pd.Timestamp(current_date)

        current_date += timedelta(days=1)

    return pd.NaT


articles["trade_date"] = articles["date"].apply(get_trade_date)

same_day = (articles["date"] == articles["trade_date"]).sum()

shifted = (
    articles["trade_date"].notna()
    & (articles["date"] != articles["trade_date"])
).sum()

backward = (
    articles["trade_date"] < articles["date"]
).sum()

print("Total articles:", len(articles))
print("Missing trade_date:", articles["trade_date"].isna().sum())
print("Same-day mappings:", same_day)
print("Shifted mappings:", shifted)
print("Backward mappings:", backward)

Total articles: 3577
Missing trade_date: 0
Same-day mappings: 2676
Shifted mappings: 901
Backward mappings: 0


Articles are mapped to the same JISDOR date when an observation exists. Otherwise, they are moved forward to the next available JISDOR trading date. If no later JISDOR observation is available, the article would remain without a trade_date. In the final dataset, however, all 3,577 articles were successfully aligned.

## 4. Daily NLP Features

In [5]:
aligned_articles = articles.dropna(
    subset=["trade_date"]
).copy()

daily_features = aggregate_daily(
    aligned_articles,
    key="trade_date"
)

daily_features = (
    daily_features
    .sort_values("trade_date")
    .reset_index(drop=True)
)

print("Aligned articles:", len(aligned_articles))
print("Daily rows with news:", len(daily_features))
print("Daily columns:", len(daily_features.columns))

Aligned articles: 3577
Daily rows with news: 392
Daily columns: 66


In [6]:
trading_days = (
    jisdor[["date_clean"]]
    .rename(columns={"date_clean": "trade_date"})
    .drop_duplicates()
    .sort_values("trade_date")
    .reset_index(drop=True)
)

daily_full = trading_days.merge(
    daily_features,
    on="trade_date",
    how="left"
)

daily_full["has_news"] = daily_full["news_count"].notna().astype(int)

feature_cols_daily = [
    col for col in daily_full.columns
    if col != "trade_date"
]

daily_full[feature_cols_daily] = (
    daily_full[feature_cols_daily]
    .fillna(0)
)

daily_full["news_count"] = daily_full["news_count"].astype(int)
daily_full["has_news"] = daily_full["has_news"].astype(int)

print("Total trading days:", len(daily_full))
print("Days with news:", daily_full["has_news"].sum())
print("Days without news:", (daily_full["has_news"] == 0).sum())
print("Missing values:", daily_full.isna().sum().sum())

daily_full.head()

Total trading days: 1202
Days with news: 392
Days without news: 810
Missing values: 0


,trade_date,news_count,title_vader_mean,title_vader_min,title_vader_std,title_neg_share,body_lm_negative,body_lm_positive,body_lm_uncertainty,body_lm_litigious,...,body_svd_25,body_svd_26,body_svd_27,body_svd_28,body_svd_29,gdelt_goldstein_mean,gdelt_goldstein_min,gdelt_tone_wmean,gdelt_mentions_sum,has_news
0,2021-09-01,25,-0.244308,-0.8357,0.366086,0.560000,0.018482,0.005959,0.006690,0.007744,...,-0.002543,0.009778,0.019143,0.028413,0.008542,-6.304000,-9.2,-4.683023,71.0,1
1,2021-09-02,19,-0.265568,-0.7351,0.282274,0.578947,0.028522,0.003982,0.005232,0.006701,...,0.017574,0.008650,0.017656,0.004555,0.009583,-6.131579,-10.0,-4.758676,20.0,1
2,2021-09-03,21,-0.271410,-0.8689,0.429169,0.619048,0.031626,0.004129,0.004951,0.009378,...,0.001180,0.009523,0.011022,-0.006186,-0.000256,-6.652381,-10.0,-4.927398,24.0,1
3,2021-09-06,15,-0.136073,-0.7430,0.325956,0.400000,0.025747,0.006840,0.006057,0.008290,...,-0.015901,-0.018708,-0.001679,0.011257,-0.014232,-6.820000,-10.0,-3.079506,19.0,1
4,2021-09-07,0,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0


The aggregated NLP features were expanded to the full JISDOR trading calendar. Trading days without sampled news were retained by filling the NLP feature values with zero and setting has_news = 0. This preserves the full financial time series while distinguishing no-news days from days with observed news features.

## 5. News and Financial Integration

In [7]:
financial["date"] = pd.to_datetime(
    financial["date"],
    errors="coerce"
).dt.normalize()

financial_merge = financial.rename(
    columns={"date": "trade_date"}
).copy()

combined = financial_merge.merge(
    daily_full,
    on="trade_date",
    how="left",
    validate="one_to_one"
)

combined = (
    combined
    .sort_values("trade_date")
    .reset_index(drop=True)
)

nlp_cols = [
    col for col in daily_full.columns
    if col != "trade_date"
]

print("Combined rows:", len(combined))
print("Combined columns:", len(combined.columns))
print("Duplicate dates:", combined["trade_date"].duplicated().sum())
print("Rows missing NLP features:", combined[nlp_cols].isna().any(axis=1).sum())
print("First date:", combined["trade_date"].min())
print("Last date:", combined["trade_date"].max())

Combined rows: 1181
Combined columns: 78
Duplicate dates: 0
Rows missing NLP features: 0
First date: 2021-09-29 00:00:00
Last date: 2026-08-31 00:00:00


The daily NLP feature table was merged with the financial modeling dataset using the final trade_date. The integration preserved all 1,181 financial modeling rows, with no duplicate dates and no missing NLP features.

## 6. Chronological Split

In [8]:
train = combined[
    combined["trade_date"] <= "2024-12-31"
].copy()

validation = combined[
    (combined["trade_date"] >= "2025-01-01")
    & (combined["trade_date"] <= "2026-03-31")
].copy()

test = combined[
    combined["trade_date"] >= "2026-04-01"
].copy()

print("Train:", len(train), train["trade_date"].min(), "to", train["trade_date"].max())
print("Validation:", len(validation), validation["trade_date"].min(), "to", validation["trade_date"].max())
print("Test:", len(test), test["trade_date"].min(), "to", test["trade_date"].max())

print("\nTrain target:")
print(train["target"].value_counts())

print("\nValidation target:")
print(validation["target"].value_counts())

print("\nTest target:")
print(test["target"].value_counts())

Train: 790 2021-09-29 00:00:00 to 2024-12-31 00:00:00
Validation: 292 2025-01-02 00:00:00 to 2026-03-31 00:00:00
Test: 99 2026-04-01 00:00:00 to 2026-08-31 00:00:00

Train target:
target
UP      444
DOWN    346
Name: count, dtype: int64

Validation target:
target
UP      158
DOWN    134
Name: count, dtype: int64

Test target:
target
UP      54
DOWN    45
Name: count, dtype: int64


The final dataset was split chronologically to prevent time-series leakage. The training period covers September 2021 to December 2024, the validation period covers January 2025 to March 2026, and the test period covers April 2026 to August 2026. No random shuffling was applied.

## 7. Combined Model Experiment

In [11]:
exclude_cols = ["trade_date", "usd_idr", "target"]

feature_cols = [
    col for col in combined.columns
    if col not in exclude_cols
]

X_train = train[feature_cols]
X_val = validation[feature_cols]
X_test = test[feature_cols]

y_train = (train["target"] == "UP").astype(int)
y_val = (validation["target"] == "UP").astype(int)
y_test = (test["target"] == "UP").astype(int)

print("Number of features:", len(feature_cols))
print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Number of features: 75
Train shape: (790, 75)
Validation shape: (292, 75)
Test shape: (99, 75)


In [12]:
def evaluate_model(y_true, y_pred, y_prob):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro F1": f1_score(y_true, y_pred, average="macro"),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob)
    }

The combined model uses 75 financial and NLP features. The dataset remains chronologically split into 790 training rows, 292 validation rows, and 99 test rows. The prediction target is the next trading-day direction (UP or DOWN).

### Logistic Regression

In [13]:
combined_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

combined_lr.fit(X_train, y_train)

val_pred_lr = combined_lr.predict(X_val)
val_prob_lr = combined_lr.predict_proba(X_val)[:, 1]

test_pred_lr = combined_lr.predict(X_test)
test_prob_lr = combined_lr.predict_proba(X_test)[:, 1]

val_results_lr = evaluate_model(
    y_val,
    val_pred_lr,
    val_prob_lr
)

test_results_lr = evaluate_model(
    y_test,
    test_pred_lr,
    test_prob_lr
)

print("Validation Results")
for metric, value in val_results_lr.items():
    print(f"{metric}: {value:.4f}")

print("\nTest Results")
for metric, value in test_results_lr.items():
    print(f"{metric}: {value:.4f}")

Validation Results
Accuracy: 0.5479
Macro F1: 0.5187
MCC: 0.0684
ROC-AUC: 0.5074

Test Results
Accuracy: 0.5455
Macro F1: 0.5261
MCC: 0.0660
ROC-AUC: 0.5333


The combined Logistic Regression achieved a test accuracy of 54.55%, a Macro F1-score of 0.5261, an MCC of 0.0660, and a ROC-AUC of 0.5333. Compared with the financial-only Logistic Regression baseline, which achieved 49.49% accuracy and a Macro F1-score of 0.4228, the combined model improved across all reported evaluation metrics. However, the relatively low MCC and ROC-AUC indicate that the predictive signal remains limited.

### XGBoost

In [14]:
combined_xgb = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

combined_xgb.fit(X_train, y_train)

val_pred_xgb = combined_xgb.predict(X_val)
val_prob_xgb = combined_xgb.predict_proba(X_val)[:, 1]

test_pred_xgb = combined_xgb.predict(X_test)
test_prob_xgb = combined_xgb.predict_proba(X_test)[:, 1]

val_results_xgb = evaluate_model(
    y_val,
    val_pred_xgb,
    val_prob_xgb
)

test_results_xgb = evaluate_model(
    y_test,
    test_pred_xgb,
    test_prob_xgb
)

print("Validation Results")
for metric, value in val_results_xgb.items():
    print(f"{metric}: {value:.4f}")

print("\nTest Results")
for metric, value in test_results_xgb.items():
    print(f"{metric}: {value:.4f}")

Validation Results
Accuracy: 0.5308
Macro F1: 0.5148
MCC: 0.0403
ROC-AUC: 0.5296

Test Results
Accuracy: 0.5152
Macro F1: 0.4875
MCC: -0.0040
ROC-AUC: 0.4802


The combined XGBoost model achieved a test accuracy of 51.52%, a Macro F1-score of 0.4875, an MCC of -0.0040, and a ROC-AUC of 0.4802. Compared with the financial-only XGBoost baseline, the combined model improved in accuracy, Macro F1-score, and MCC, but its ROC-AUC decreased. This suggests that the additional NLP features did not consistently improve the ranking performance of the XGBoost model.

## 8. Baseline vs Combined Results

In [15]:
comparison = pd.DataFrame({
    "Model": [
        "Financial-only Logistic Regression",
        "Combined Logistic Regression",
        "Financial-only XGBoost",
        "Combined XGBoost"
    ],
    "Accuracy": [
        0.4949,
        test_results_lr["Accuracy"],
        0.4646,
        test_results_xgb["Accuracy"]
    ],
    "Macro F1": [
        0.4228,
        test_results_lr["Macro F1"],
        0.4312,
        test_results_xgb["Macro F1"]
    ],
    "MCC": [
        -0.0843,
        test_results_lr["MCC"],
        -0.1164,
        test_results_xgb["MCC"]
    ],
    "ROC-AUC": [
        0.5004,
        test_results_lr["ROC-AUC"],
        0.4947,
        test_results_xgb["ROC-AUC"]
    ]
})

comparison

,Model,Accuracy,Macro F1,MCC,ROC-AUC
0,Financial-only Logistic Regression,0.494900,0.422800,-0.084300,0.500400
1,Combined Logistic Regression,0.545455,0.526114,0.066023,0.533333
2,Financial-only XGBoost,0.464600,0.431200,-0.116400,0.494700
3,Combined XGBoost,0.515152,0.487489,-0.003977,0.480247


The combined Logistic Regression model outperformed its financial-only baseline across all test metrics. The combined XGBoost model improved in accuracy, Macro F1-score, and MCC, but not ROC-AUC. Overall, the NLP features appear to add some predictive value, although the overall predictive signal remains limited.

## 9. Save Outputs

In [16]:
aligned_articles.to_csv(
    BASE_DIR / "article_features_aligned.csv",
    index=False
)

daily_full.to_csv(
    BASE_DIR / "daily_nlp_features.csv",
    index=False
)

combined.to_csv(
    BASE_DIR / "combined_dataset.csv",
    index=False
)

train.to_csv(
    BASE_DIR / "train.csv",
    index=False
)

validation.to_csv(
    BASE_DIR / "validation.csv",
    index=False
)

test.to_csv(
    BASE_DIR / "test.csv",
    index=False
)

comparison.to_csv(
    BASE_DIR / "baseline_vs_combined_results.csv",
    index=False
)

with open(BASE_DIR / "combined_lr_results.json", "w") as f:
    json.dump(
        {
            "model": "Logistic Regression",
            "validation": val_results_lr,
            "test": test_results_lr
        },
        f,
        indent=4
    )

with open(BASE_DIR / "combined_xgb_results.json", "w") as f:
    json.dump(
        {
            "model": "XGBoost",
            "validation": val_results_xgb,
            "test": test_results_xgb
        },
        f,
        indent=4
    )

print("Saved final Task 2 outputs.")

Saved final Task 2 outputs.
